In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/telco_churn.csv')

print(f"Raw shape: {df.shape}")

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"TotalCharges nulls after fix: {df['TotalCharges'].isnull().sum()}")

df = df.dropna(subset=['TotalCharges'])
print(f"After dropping nulls: {df.shape}")

Raw shape: (7043, 21)
TotalCharges nulls after fix: 11
After dropping nulls: (7032, 21)


In [2]:
df = df.drop(columns=['customerID'])

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(df['Churn'].value_counts())
print(f"Churn rate: {df['Churn'].mean()*100:.2f}%")

Churn
0    5163
1    1869
Name: count, dtype: int64
Churn rate: 26.58%


In [3]:
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'MultipleLines', 'OnlineSecurity',
               'OnlineBackup', 'DeviceProtection', 'TechSupport',
               'StreamingTV', 'StreamingMovies']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0,
                            'Male': 1, 'Female': 0,
                            'No phone service': 0,
                            'No internet service': 0})

print(df[binary_cols].head())
print(df[binary_cols].dtypes)

   gender  Partner  Dependents  PhoneService  PaperlessBilling  MultipleLines  \
0       0        1           0             0                 1              0   
1       1        0           0             1                 0              0   
2       1        0           0             1                 1              0   
3       1        0           0             0                 0              0   
4       0        0           0             1                 1              0   

   OnlineSecurity  OnlineBackup  DeviceProtection  TechSupport  StreamingTV  \
0               0             1                 0            0            0   
1               1             0                 1            0            0   
2               1             1                 0            0            0   
3               1             0                 1            1            0   
4               0             0                 0            0            0   

   StreamingMovies  
0                

In [4]:
multi_cols = ['InternetService', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)
print(df.shape)
print(df.columns.tolist())

(7032, 24)
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [5]:
df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'] + 1)
df['ChargePerService'] = df['MonthlyCharges'] / (df[binary_cols].sum(axis=1) + 1)
df['TenureGroup'] = pd.cut(df['tenure'],
                            bins=[0, 12, 24, 48, 72],
                            labels=['New', 'Growing', 'Mature', 'Loyal'])
df['TenureGroup'] = df['TenureGroup'].astype(str)
df = pd.get_dummies(df, columns=['TenureGroup'], drop_first=True)

print(f"Final shape: {df.shape}")
print(df.head(3))

Final shape: (7032, 29)
   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       0              0        1           0       1             0   
1       1              0        0           0      34             1   
2       1              0        0           0       2             1   

   MultipleLines  OnlineSecurity  OnlineBackup  DeviceProtection  ...  \
0              0               0             1                 0  ...   
1              0               1             0                 1  ...   
2              0               1             1                 0  ...   

   Contract_One year  Contract_Two year  \
0              False              False   
1               True              False   
2              False              False   

   PaymentMethod_Credit card (automatic)  PaymentMethod_Electronic check  \
0                                  False                            True   
1                                  False                           False 

In [6]:
corr = df.corr()['Churn'].sort_values(ascending=False)
print("Top features correlated with churn:")
print(corr.head(10))
print("\nLeast correlated:")
print(corr.tail(5))

Top features correlated with churn:
Churn                             1.000000
ChargePerService                  0.353091
TenureGroup_New                   0.319628
InternetService_Fiber optic       0.307463
PaymentMethod_Electronic check    0.301455
MonthlyCharges                    0.192858
PaperlessBilling                  0.191454
SeniorCitizen                     0.150541
AvgMonthlyCharge                  0.070992
StreamingTV                       0.063254
Name: Churn, dtype: float64

Least correlated:
TotalCharges         -0.199484
InternetService_No   -0.227578
TenureGroup_Loyal    -0.264035
Contract_Two year    -0.301552
tenure               -0.354049
Name: Churn, dtype: float64


In [7]:
df.to_csv('../data/processed/churn_clean.csv', index=False)
print(f"Saved. Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Saved. Shape: (7032, 29)
Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'AvgMonthlyCharge', 'ChargePerService', 'TenureGroup_Loyal', 'TenureGroup_Mature', 'TenureGroup_New']
